In [1]:
%%capture
!pip -q install -U pyspark pyarrow scikit-learn matplotlib huggingface_hub


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content")

DATA_DIR = PROJECT_ROOT / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

TABLEAU_DIR = GOLD_DIR / "tableau_csv"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
METRICS_DIR = OUTPUTS_DIR / "metrics"

HF_CACHE_DIR = PROJECT_ROOT / ".hf_cache"
HF_DATASET_DIR = PROJECT_ROOT / "hf_dataset"

for p in [BRONZE_DIR, SILVER_DIR, GOLD_DIR, TABLEAU_DIR, FIGURES_DIR, METRICS_DIR, HF_CACHE_DIR, HF_DATASET_DIR]:
    p.mkdir(parents=True, exist_ok=True)

## 1 Spark session


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CiferFraudColab")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.local.dir", str(OUTPUTS_DIR / "spark_local_dir"))
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## 2 Download dataset files (raw CSV)


In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import glob

DATASET_REPO = "CiferAI/Cifer-Fraud-Detection-Dataset-AF"

snapshot_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    cache_dir=str(HF_CACHE_DIR),
    local_dir=str(HF_DATASET_DIR),
    local_dir_use_symlinks=False,
)

csv_files = sorted(glob.glob(str(HF_DATASET_DIR / "**" / "*.csv"), recursive=True))
if len(csv_files) == 0:
    raise RuntimeError("CSV files not found in hf_dataset")


## 3 Bronze (RDD raw ingestion -> Parquet)


In [ ]:
import csv
from io import StringIO
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql import functions as F

schema = StructType([
    StructField("step", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("nameOrig", StringType(), True),
    StructField("oldbalanceOrg", DoubleType(), True),
    StructField("newbalanceOrig", DoubleType(), True),
    StructField("nameDest", StringType(), True),
    StructField("oldbalanceDest", DoubleType(), True),
    StructField("newbalanceDest", DoubleType(), True),
    StructField("isFraud", IntegerType(), True),
    StructField("isFlaggedFraud", IntegerType(), True),
])

HEADER_PREFIX = "step,"

def parse_line(line):
    try:
        fields = next(csv.reader(StringIO(line)))
        if len(fields) != 11:
            return None
        return Row(
            step=int(fields[0]),
            type=fields[1],
            amount=float(fields[2]),
            nameOrig=fields[3],
            oldbalanceOrg=float(fields[4]),
            newbalanceOrig=float(fields[5]),
            nameDest=fields[6],
            oldbalanceDest=float(fields[7]),
            newbalanceDest=float(fields[8]),
            isFraud=int(fields[9]),
            isFlaggedFraud=int(fields[10]),
        )
    except Exception:
        return None

rdd = spark.sparkContext.textFile(",".join(csv_files))
rdd = rdd.filter(lambda x: x and not x.startswith(HEADER_PREFIX))
good = rdd.map(parse_line).filter(lambda x: x is not None)

df_bronze = spark.createDataFrame(good, schema=schema)

df_bronze = (
    df_bronze.withColumn("ingestion_ts", F.current_timestamp())
             .withColumn("step_day", (F.col("step") / F.lit(24)).cast("int"))
)

(df_bronze.repartition("step_day")
          .write.mode("overwrite")
          .partitionBy("step_day")
          .parquet(str(BRONZE_DIR / "transactions_parquet")))

df_bronze.select("step", "type", "amount", "isFraud").limit(5).toPandas()


## 4 Silver (cleaning + validation -> Parquet)


In [ ]:
import json
import time
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

df = spark.read.parquet(str(BRONZE_DIR / "transactions_parquet"))

qc = df.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("step").isNull().cast("int")).alias("null_step"),
    F.sum(F.col("type").isNull().cast("int")).alias("null_type"),
    F.sum((F.col("amount") < 0).cast("int")).alias("neg_amount"),
    F.sum((~F.col("isFraud").isin([0, 1])).cast("int")).alias("isFraud_not_01"),
    F.sum((~F.col("isFlaggedFraud").isin([0, 1])).cast("int")).alias("isFlaggedFraud_not_01"),
).collect()[0].asDict()

(METRICS_DIR / "silver_quality_checks.json").write_text(json.dumps(qc, indent=2))

df_silver = (
    df.withColumn("type", F.upper(F.trim(F.col("type"))))
      .withColumn("amount", F.col("amount").cast("double"))
      .withColumn("oldbalanceOrg", F.col("oldbalanceOrg").cast("double"))
      .withColumn("newbalanceOrig", F.col("newbalanceOrig").cast("double"))
      .withColumn("oldbalanceDest", F.col("oldbalanceDest").cast("double"))
      .withColumn("newbalanceDest", F.col("newbalanceDest").cast("double"))
      .withColumn("isFraud", F.col("isFraud").cast("int"))
      .withColumn("isFlaggedFraud", F.col("isFlaggedFraud").cast("int"))
)

df_silver = df_silver.filter(F.col("amount") >= 0)
df_silver = df_silver.filter(F.col("isFraud").isin([0, 1]) & F.col("isFlaggedFraud").isin([0, 1]))

df_silver = (
    df_silver.withColumn("org_balance_delta", F.col("newbalanceOrig") - F.col("oldbalanceOrg"))
             .withColumn("dest_balance_delta", F.col("newbalanceDest") - F.col("oldbalanceDest"))
             .withColumn("hour_of_day", (F.col("step") % 24).cast("int"))
)

t0 = time.time()

(df_silver
 .write.mode("overwrite")
 .partitionBy("step_day", "type")
 .parquet(str(SILVER_DIR / "transactions_parquet")))

t1 = time.time()
(METRICS_DIR / "silver_write_seconds.json").write_text(json.dumps({"write_seconds": round(t1 - t0, 3)}, indent=2))

df_cache_sample = df_silver.sample(False, 0.01, seed=42).persist(StorageLevel.MEMORY_AND_DISK)
_ = df_cache_sample.count()
df_cache_sample.unpersist()

spark.read.parquet(str(SILVER_DIR / "transactions_parquet")).select("step", "type", "amount", "isFraud").limit(5).toPandas()

## 5 Gold


In [ ]:
import shutil
from pathlib import Path
from pyspark.sql import functions as F

df = spark.read.parquet(str(SILVER_DIR / "transactions_parquet"))

gold_volume = (
    df.groupBy("step_day", "type")
      .agg(
          F.count("*").alias("txn_count"),
          F.sum(F.col("isFraud")).alias("fraud_count"),
          (F.sum(F.col("isFraud")) / F.count("*")).alias("fraud_rate"),
          F.sum("amount").alias("total_amount"),
          F.avg("amount").alias("avg_amount"),
      )
)

gold_hourly = (
    df.groupBy("hour_of_day", "type")
      .agg(
          F.count("*").alias("txn_count"),
          F.sum(F.col("isFraud")).alias("fraud_count"),
          (F.sum(F.col("isFraud")) / F.count("*")).alias("fraud_rate"),
          F.avg("amount").alias("avg_amount"),
      )
)

gold_amount_bands = (
    df.withColumn(
        "amount_band",
        F.when(F.col("amount") < 10, "<10")
         .when(F.col("amount") < 100, "10-99")
         .when(F.col("amount") < 1000, "100-999")
         .when(F.col("amount") < 10000, "1k-9.9k")
         .otherwise(">=10k")
    )
    .groupBy("amount_band", "type")
    .agg(
        F.count("*").alias("txn_count"),
        F.sum("isFraud").alias("fraud_count"),
        (F.sum("isFraud") / F.count("*")).alias("fraud_rate"),
        F.avg("amount").alias("avg_amount"),
    )
)

cols = ["step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig",
        "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud"]

total = df.count()
exprs = [(F.sum(F.col(c).isNull().cast("int")) / F.lit(total)).alias(f"null_rate_{c}") for c in cols]
gold_quality = df.agg(*exprs)

def write_gold_parquet(df_in, name):
    df_in.write.mode("overwrite").parquet(str(GOLD_DIR / name / "parquet"))

def write_single_csv(df_in, out_file: Path):
    tmp_dir = out_file.parent / (out_file.stem + "_tmp")
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    (df_in.coalesce(1)
         .write.mode("overwrite")
         .option("header", True)
         .csv(str(tmp_dir)))
    part = next(tmp_dir.glob("part-*.csv"))
    if out_file.exists():
        out_file.unlink()
    part.replace(out_file)
    shutil.rmtree(tmp_dir)

write_gold_parquet(gold_volume, "gold_volume_by_day_type")
write_gold_parquet(gold_hourly, "gold_hourly_by_type")
write_gold_parquet(gold_amount_bands, "gold_amount_bands_by_type")
write_gold_parquet(gold_quality, "gold_quality_null_rates")

write_single_csv(gold_volume, TABLEAU_DIR / "gold_volume_by_day_type.csv")
write_single_csv(gold_hourly, TABLEAU_DIR / "gold_hourly_by_type.csv")
write_single_csv(gold_amount_bands, TABLEAU_DIR / "gold_amount_bands_by_type.csv")
write_single_csv(gold_quality, TABLEAU_DIR / "gold_quality_null_rates.csv")

gold_volume.orderBy("step_day", "type").limit(10).toPandas()


## 6 ML pipeline (Spark MLlib) + model comparison table


In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Transformer
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.sql import DataFrame

TRAIN_SAMPLE_FRACTION = 0.10

df = spark.read.parquet(str(SILVER_DIR / "transactions_parquet")).sample(False, TRAIN_SAMPLE_FRACTION, seed=42)

label_col = "isFraud"

class DomainFeatureEngineer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    def _transform(self, dataset: DataFrame) -> DataFrame:
        return (
            dataset
            .withColumn("amount_log1p", F.log1p(F.col("amount")))
            .withColumn("abs_org_delta", F.abs(F.col("org_balance_delta")))
            .withColumn("abs_dest_delta", F.abs(F.col("dest_balance_delta")))
            .withColumn("is_zero_newbalanceOrig", (F.col("newbalanceOrig") == 0).cast("int"))
        )

feat_eng = DomainFeatureEngineer()

type_indexer = StringIndexer(inputCol="type", outputCol="type_idx", handleInvalid="keep")
type_ohe = OneHotEncoder(inputCols=["type_idx"], outputCols=["type_ohe"])

numeric_cols = [
    "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
    "org_balance_delta", "dest_balance_delta", "hour_of_day", "isFlaggedFraud",
    "amount_log1p", "abs_org_delta", "abs_dest_delta", "is_zero_newbalanceOrig"
]

assembler = VectorAssembler(inputCols=["type_ohe"] + numeric_cols, outputCol="features_raw", handleInvalid="keep")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)

base_stages = [feat_eng, type_indexer, type_ohe, assembler, scaler]

train, test = df.randomSplit([0.8, 0.2], seed=42)

evaluator_pr = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderPR")
evaluator_roc = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")

models = {
    "LR": LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=30),
    "DT": DecisionTreeClassifier(featuresCol="features", labelCol=label_col, maxDepth=8),
    "RF": RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=80, maxDepth=10),
    "GBT": GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5),
}

fitted = {}
rows = []

for name, clf in models.items():
    pipe = Pipeline(stages=base_stages + [clf])
    t0 = time.time()
    m = pipe.fit(train)
    t1 = time.time()
    pred = m.transform(test)
    pr = float(evaluator_pr.evaluate(pred))
    roc = float(evaluator_roc.evaluate(pred))
    rows.append({"model": name, "PR_AUC": pr, "ROC_AUC": roc, "fit_seconds": float(t1-t0)})
    fitted[name] = m

metrics_df = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
(METRICS_DIR / "spark_mllib_metrics.json").write_text(metrics_df.to_json(orient="records", indent=2))
metrics_df


## 7 Significance tests + curves + confusion matrix


In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    roc_curve, precision_recall_curve,
    confusion_matrix
)

best_two = metrics_df.loc[:1, "model"].tolist()

eval_df = test.sample(False, 0.20, seed=7).limit(200000)

def get_scores(model_name):
    p = fitted[model_name].transform(eval_df).select(
        F.col(label_col).cast("int").alias("y"),
        vector_to_array("probability")[1].alias("p")
    )
    return p.toPandas()

pdf_a = get_scores(best_two[0])
pdf_b = get_scores(best_two[1])

y = pdf_a["y"].to_numpy()
pa = pdf_a["p"].to_numpy()
pb = pdf_b["p"].to_numpy()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def paired_bootstrap_pvalue(y_true, p1, p2, metric_fn, n_boot=300, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    idx = np.arange(n)
    diffs = []
    for _ in range(n_boot):
        b = rng.choice(idx, size=n, replace=True)
        diffs.append(metric_fn(y_true[b], p1[b]) - metric_fn(y_true[b], p2[b]))
    diffs = np.array(diffs)
    p = float(2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    return float(diffs.mean()), float(np.quantile(diffs, 0.025)), float(np.quantile(diffs, 0.975)), p

pr_mean, pr_lo, pr_hi, pr_p = paired_bootstrap_pvalue(y, pa, pb, average_precision_score, n_boot=300)
roc_mean, roc_lo, roc_hi, roc_p = paired_bootstrap_pvalue(y, pa, pb, roc_auc_score, n_boot=300)

sig = pd.DataFrame([
    {"metric": "PR_AUC_diff", "mean_diff": pr_mean, "ci_low": pr_lo, "ci_high": pr_hi, "p_value": pr_p},
    {"metric": "ROC_AUC_diff", "mean_diff": roc_mean, "ci_low": roc_lo, "ci_high": roc_hi, "p_value": roc_p},
])
(METRICS_DIR / "paired_bootstrap_significance.json").write_text(sig.to_json(orient="records", indent=2))

thr = 0.5
cm = confusion_matrix(y, (pa >= thr).astype(int))

fig = plt.figure()
ax = fig.add_subplot(111)
ax.imshow(cm)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=200)
plt.show()

fpr, tpr, _ = roc_curve(y, pa)
fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(fpr, tpr)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "roc_curve.png", dpi=200)
plt.show()

prec, rec, _ = precision_recall_curve(y, pa)
fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(rec, prec)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "pr_curve.png", dpi=200)
plt.show()

sig

## 8 Learning curve + feature importance plots


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

base_pdf = (
    spark.read.parquet(str(SILVER_DIR / "transactions_parquet"))
         .sample(False, 0.01, seed=10)
         .select("type", "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
                 "org_balance_delta", "dest_balance_delta", "hour_of_day", "isFlaggedFraud", "isFraud")
         .limit(300000)
         .toPandas()
)

X = base_pdf.drop(columns=["isFraud"])
y2 = base_pdf["isFraud"].astype(int)

cat_cols = ["type"]
num_cols = [c for c in X.columns if c not in cat_cols]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])

Xt, Xv, yt, yv = train_test_split(X, y2, test_size=0.2, random_state=42, stratify=y2)

sizes = [20000, 50000, 100000, 200000]
scores = []
for s in sizes:
    m = SkPipeline(steps=[("pre", pre), ("clf", SkLR(max_iter=300, n_jobs=-1))])
    m.fit(Xt.iloc[:s], yt.iloc[:s])
    p = m.predict_proba(Xv)[:, 1]
    scores.append(float(average_precision_score(yv, p)))

lc = pd.DataFrame({"train_size": sizes, "PR_AUC": scores})
(METRICS_DIR / "learning_curve_pr_auc.json").write_text(lc.to_json(orient="records", indent=2))

fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(lc["train_size"], lc["PR_AUC"])
ax.set_xlabel("Train size")
ax.set_ylabel("PR-AUC")
ax.set_title("Learning Curve (PR-AUC)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "learning_curve.png", dpi=200)
plt.show()

rf = SkPipeline(steps=[("pre", pre), ("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42))])
rf.fit(Xt, yt)

try:
    feature_names = rf.named_steps["pre"].get_feature_names_out()
except Exception:
    feature_names = np.array([f"f{i}" for i in range(rf.named_steps["clf"].feature_importances_.shape[0])])

imp = rf.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"feature": feature_names, "importance": imp}).sort_values("importance", ascending=False).head(20)
(METRICS_DIR / "feature_importance_top20.json").write_text(imp_df.to_json(orient="records", indent=2))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
ax.barh(imp_df["feature"][::-1], imp_df["importance"][::-1])
ax.set_xlabel("Importance")
ax.set_title("Feature Importance (Top 20)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "feature_importance.png", dpi=200)
plt.show()

imp_df
